In [142]:
# Data Processing
import pandas as pd
import numpy as np

# Modelling
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint

# Tree Visualisation
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

from sklearn.model_selection import ParameterGrid, StratifiedKFold
from tqdm import tqdm

In [189]:
df = pd.read_csv('https://raw.githubusercontent.com/lliwpugs2/Seizure-prediction-research-using-chemical-detection/refs/heads/main/AllTrialsNoHT.csv')
df

,mq2,mq3,mq4,mq6,mq7,mq8,mq9,mq135,presence
0,213,281,404,452,524,483,579,573,0
1,213,281,404,453,525,483,580,574,0
2,213,281,404,453,525,483,580,574,0
3,213,281,404,453,525,483,581,574,0
4,212,281,404,453,525,484,581,575,0
...,...,...,...,...,...,...,...,...,...
7510,160,282,381,456,442,441,511,533,0
7511,161,282,380,456,442,441,510,532,0
7512,161,282,380,456,442,441,511,532,0
7513,161,282,380,456,442,441,511,532,0


In [190]:
y = df['presence']
y

0       0
1       0
2       0
3       0
4       0
       ..
7510    0
7511    0
7512    0
7513    0
7514    0
Name: presence, Length: 7515, dtype: int64

In [191]:
X = df.drop('presence', axis=1)
X

,mq2,mq3,mq4,mq6,mq7,mq8,mq9,mq135
0,213,281,404,452,524,483,579,573
1,213,281,404,453,525,483,580,574
2,213,281,404,453,525,483,580,574
3,213,281,404,453,525,483,581,574
4,212,281,404,453,525,484,581,575
...,...,...,...,...,...,...,...,...
7510,160,282,381,456,442,441,511,533
7511,161,282,380,456,442,441,510,532
7512,161,282,380,456,442,441,511,532
7513,161,282,380,456,442,441,511,532


In [192]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

In [193]:
X_train

,mq2,mq3,mq4,mq6,mq7,mq8,mq9,mq135
4779,185,280,380,446,474,444,487,548
5321,219,292,398,469,505,433,507,551
420,189,275,394,440,515,481,570,558
5363,217,293,397,469,503,433,505,549
6289,145,292,370,434,464,462,494,547
...,...,...,...,...,...,...,...,...
79,209,280,403,451,523,483,579,571
3927,141,286,375,439,435,456,472,543
5955,209,272,388,466,477,457,496,537
6936,191,301,390,484,442,464,516,543


In [194]:
# set noise level (standard deviation multiplier)
noise_level = 0.5

# convert DataFrame to float to avoid type issues
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# add Gaussian noise to both train and test
X_train += np.random.normal(0, noise_level * X_train.std(axis=0), X_train.shape)
X_test += np.random.normal(0, noise_level * X_test.std(axis=0), X_test.shape)

# print the noisy training data
print(X_train)

             mq2         mq3         mq4         mq6         mq7         mq8  \
4779  189.691816  297.246998  382.468867  437.320339  468.068614  441.742500   
5321  215.617508  295.014336  396.176129  477.656597  517.947820  439.147561   
420   172.985245  268.276217  384.691765  438.871826  510.872650  482.406450   
5363  204.676479  306.698423  397.061787  472.342805  489.575587  427.852845   
6289  150.622212  315.632778  369.911664  440.881139  449.562018  460.757547   
...          ...         ...         ...         ...         ...         ...   
79    208.188540  274.037155  397.625653  452.570844  530.864466  480.188470   
3927  104.349263  268.256873  368.695166  436.830161  427.991283  463.790307   
5955  206.447289  265.477507  389.485707  466.614292  496.442205  459.647486   
6936  193.088275  321.379317  390.807898  510.927155  428.716107  448.538105   
5640  214.326289  259.073607  396.418086  433.631024  474.165946  465.214393   

             mq9       mq135  
4779  47

In [195]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [196]:
y_pred = rf.predict(X_test)

In [197]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
# accuracy before tuning hyperparameters

Accuracy: 0.9308050565535595


In [152]:
# Complete parameter grid including n_estimators
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 4, 8, 16]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
grid = list(ParameterGrid(param_grid))

with tqdm(total=len(grid), desc="Grid Search Progress", unit="combo") as pbar:
    for params in grid:
        train_scores = []
        val_scores = []

        for train_idx, val_idx in cv.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

            model = RandomForestClassifier(
                **params,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_tr, y_tr)

            train_scores.append(
                accuracy_score(y_tr, model.predict(X_tr))
            )
            val_scores.append(
                accuracy_score(y_val, model.predict(X_val))
            )

        results.append({
            **params,
            'train_accuracy': np.mean(train_scores),
            'val_accuracy': np.mean(val_scores),
            'overfit_gap': np.mean(train_scores) - np.mean(val_scores)
        })

        pbar.update(1)

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# ✅ FIX: prioritize validation accuracy, lightly penalize overfitting
results_df['balanced_score'] = (
    results_df['val_accuracy'] - 0.1 * results_df['overfit_gap']
)

# Select best-performing row
best_row = results_df.sort_values(
    'balanced_score', ascending=False
).iloc[0]

# --- FIXED: Properly handle None and NaN values when casting parameters ---
best_params = {}
for key in param_grid.keys():
    value = best_row[key]
    if value is None or pd.isna(value):
        best_params[key] = None
    else:
        best_params[key] = int(value)

# Train final model
final_model = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# Optional: view chosen params
print("Best hyperparameters:", best_params)

Grid Search Progress: 100%|██████████| 625/625 [11:43:13<00:00, 67.51s/combo]      


Best hyperparameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 1}


In [159]:
best_params = {
    'n_estimators': 100,
    'max_depth': None,
    'min_samples_split': 2,
    'min_samples_leaf': 1
}

rf = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [160]:
y_pred = rf.predict(X_test)

In [161]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9321357285429142


In [41]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Metrics
accuracy = accuracy_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)      # same as recall
precision = precision_score(y_test, y_pred)
specificity = tn / (tn + fp)

# Print results
print("Accuracy:", accuracy)
print("Sensitivity (Recall):", sensitivity)
print("Specificity:", specificity)
print("Precision:", precision)

Accuracy: 0.9201596806387226
Sensitivity (Recall): 0.8670634920634921
Specificity: 0.9469469469469469
Precision: 0.8918367346938776


In [16]:
# Create a single-row DataFrame with the SAME columns
new_data = pd.DataFrame([{
    "mq2": 386,
    "mq3": 385,
    "mq4": 384,
    "mq5": 381,
    "mq6": 372,
    "mq7": 373,
    "mq8": 373,
    "mq9": 375,
    "mq135": 376,
    # ...
}])

prediction = rf.predict(new_data)
print(prediction)

[1]
